In [1]:
"""
Pipeline completo:
 - Leer lecturas desde LecturasBalanceBoard (SQL Server)
 - Por cada par (UsuarioId, NumeroPruebas):
    * ordenamos por TimeStamp
    * calculamos t (segundos relativos)
    * interpolamos a 100 Hz TopLeft/TopRight/BottomLeft/BottomRight/COP_X/COP_Y/Total
    * guardamos interpolados en LecturasInterpoladas
    * aplicamos Butterworth 4th order fc=10Hz (filtfilt) sobre COP_X,COP_Y
    * guardamos en LecturasConFiltro (mismas columnas, COP's filtrados)
    * aplicamos media móvil 1s (100 muestras) sobre COP_X_filt,COP_Y_filt
    * guardamos en LecturasConMediaMovil
    * calculamos métricas (excluyendo primer 1s y último 1s) y guardamos en Evaluaciones
"""

import pyodbc
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import butter, filtfilt
from datetime import timedelta
import math
import sys

# ========== Configuración ==========
CONN_STR = (
    r"DRIVER={SQL Server};"
    r"SERVER=PP-WALL-E\SQLEXPRESS;"
    r"DATABASE=Ultrasound;"
    r"Trusted_Connection=yes;"
)

FS = 100.0         # Frecuencia objetivo (Hz)
DT = 1.0 / FS      # Paso temporal (s)
FC = 10.0          # Frecuencia de corte Butterworth (Hz)
ORDER = 4          # Orden del filtro Butterworth
MOVING_WINDOW = int(FS * 1.0)  # 1 s -> 100 muestras

# ========== Funciones auxiliares ==========

def connect():
    return pyodbc.connect(CONN_STR)

def read_groups(conn):
    """
    Leer todas las filas desde LecturasBalanceBoard y agrupar por UsuarioId, NumeroPruebas.
    Devuelve una lista de tuplas (UsuarioId, NumeroPruebas, df_group)
    """
    query = """
    SELECT UsuarioId, NumeroPruebas, TopLeft, TopRight, BottomLeft, BottomRight, COP_X, COP_Y, Total, TimeStamp
    FROM LecturasBalanceBoard
    ORDER BY UsuarioId, NumeroPruebas, TimeStamp
    """
    df = pd.read_sql(query, conn)
    if df.empty:
        return []
    # Asegurar tipo datetime
    df['TimeStamp'] = pd.to_datetime(df['TimeStamp'])
    groups = []
    for (uid, num), g in df.groupby(['UsuarioId', 'NumeroPruebas'], sort=True):
        groups.append((uid, num, g.sort_values('TimeStamp').reset_index(drop=True)))
    return groups

def make_uniform_time(t0, tf):
    """Genera array de tiempos desde t0 (segundos) hasta tf (inclusive aproximado) con paso DT."""
    # usamos np.arange con pequeño margen para incluir tf si cae justo
    return np.arange(t0, tf + 1e-9, DT)

def interp_series(t_orig, y_orig, t_uniform, kind='linear'):
    """Interpolar y_orig(t_orig) en los tiempos t_uniform. Devuelve array."""
    if len(t_orig) < 2:
        # No se puede interpolar: devolver NaNs
        return np.full_like(t_uniform, np.nan, dtype=float)
    f = interp1d(t_orig, y_orig, kind=kind, bounds_error=False, fill_value="extrapolate")
    return f(t_uniform)

def design_butterworth(fs, fc, order):
    nyq = 0.5 * fs
    wn = fc / nyq
    b, a = butter(order, wn, btype='low', analog=False)
    return b, a

def apply_zero_phase_filter(arr, b, a):
    """Aplica filtfilt con manejo de longitudes pequeñas."""
    # filtfilt necesita más muestras que padlen = 3*(max(len(a),len(b))).
    try:
        if np.all(np.isnan(arr)):
            return arr.copy()
        # filtfilt requiere suficiente longitud
        if len(arr) < (max(len(a), len(b)) * 3):
            # si serie demasiado corta, devolvemos la serie original (o suave con convolve)
            return arr.copy()
        return filtfilt(b, a, arr)
    except Exception as e:
        # En caso de fallo, devolvemos original y avisamos
        print("Warning: filtro no aplicado por:", e)
        return arr.copy()

def moving_average(arr, window):
    """Media móvil causal (media de las últimas 'window' muestras, min_periods=window)"""
    ser = pd.Series(arr)
    # usamos min_periods=window para que las primeras (window-1) sean NaN
    res = ser.rolling(window=window, min_periods=window).mean().to_numpy()
    return res

def insert_many(conn, table, columns, rows):
    """
    Inserta muchas filas en tabla. columns = list de nombres, rows = iterable de tuplas.
    Usamos fast executemany si disponible.
    """
    cursor = conn.cursor()
    placeholders = ", ".join(["?"] * len(columns))
    cols = ", ".join(columns)
    sql = f"INSERT INTO {table} ({cols}) VALUES ({placeholders})"
    cursor.fast_executemany = True
    cursor.executemany(sql, rows)
    conn.commit()
    cursor.close()

# ========== Pipeline por grupo ==========
def process_group(conn, usuarioId, numeroPruebas, df):
    """
    df tiene columnas: TopLeft, TopRight, BottomLeft, BottomRight, COP_X, COP_Y, Total, TimeStamp
    """
    n_orig = len(df)
    print(f"\nProcesando Usuario={usuarioId}, Prueba={numeroPruebas}, muestras originales={n_orig}")

    # 1) construir t (segundos relativos)
    t0 = df['TimeStamp'].iloc[0]
    df['t'] = (df['TimeStamp'] - t0).dt.total_seconds()

    # validación mínima de muestras
    if len(df) < 3:
        print("  -> Demasiado pocas muestras para interpolar/filtar. Se omite.")
        return

    # 2) generar tiempos uniformes
    t_uniform = make_uniform_time(df['t'].iloc[0], df['t'].iloc[-1])
    # si la duración es menor que un segundo, t_uniform será corto; toleramos

    # 3) interpolar señales (lineal)
    cols_to_interp = ['TopLeft', 'TopRight', 'BottomLeft', 'BottomRight', 'COP_X', 'COP_Y', 'Total']
    interp_data = {}
    for col in cols_to_interp:
        interp_data[col] = interp_series(df['t'].to_numpy(), df[col].to_numpy(), t_uniform, kind='linear')

    # generar timestamps interpolados reales (datetime)
    timestamps_uniform = [t0 + timedelta(seconds=float(s)) for s in t_uniform]

    # preparar filas para LecturasInterpoladas
    rows_interp = []
    for i, t_abs in enumerate(timestamps_uniform):
        rows_interp.append((
            usuarioId,
            numeroPruebas,
            float(interp_data['TopLeft'][i]),
            float(interp_data['TopRight'][i]),
            float(interp_data['BottomLeft'][i]),
            float(interp_data['BottomRight'][i]),
            float(interp_data['COP_X'][i]),
            float(interp_data['COP_Y'][i]),
            float(interp_data['Total'][i]),
            t_abs
        ))

    # insertar interpolados
    if rows_interp:
        insert_many(conn, "LecturasInterpoladas",
                    ["UsuarioId", "NumeroPruebas", "TopLeft", "TopRight", "BottomLeft", "BottomRight", "COP_X", "COP_Y", "Total", "TimeStamp"],
                    rows_interp)
    print(f"  -> Interpoladas e insertadas {len(rows_interp)} filas.")

    # 4) FILTRADO Butterworth (aplicar a la serie interpolada COP_X/COP_Y)
    b, a = design_butterworth(FS, FC, ORDER)
    copx_arr = interp_data['COP_X']
    copy_arr = interp_data['COP_Y']

    copx_filt = apply_zero_phase_filter(copx_arr, b, a)
    copy_filt = apply_zero_phase_filter(copy_arr, b, a)

    # preparar filas para LecturasConFiltro (guardamos las mismas columnas pero con COPs filtradas)
    rows_filt = []
    for i, t_abs in enumerate(timestamps_uniform):
        rows_filt.append((
            usuarioId,
            numeroPruebas,
            float(interp_data['TopLeft'][i]),
            float(interp_data['TopRight'][i]),
            float(interp_data['BottomLeft'][i]),
            float(interp_data['BottomRight'][i]),
            float(copx_filt[i]),    # COP_X filtrado
            float(copy_filt[i]),    # COP_Y filtrado
            float(interp_data['Total'][i]),
            t_abs
        ))
    if rows_filt:
        insert_many(conn, "LecturasConFiltro",
                    ["UsuarioId","NumeroPruebas","TopLeft","TopRight","BottomLeft","BottomRight","COP_X","COP_Y","Total","TimeStamp"],
                    rows_filt)
    print(f"  -> Filtrado e insertadas {len(rows_filt)} filas en LecturasConFiltro.")

    # 5) MEDIA MÓVIL 1 s (ventana causal de 100 muestras): aplicamos sobre COP filtrado
    copx_mov = moving_average(copx_filt, MOVING_WINDOW)
    copy_mov = moving_average(copy_filt, MOVING_WINDOW)

    # Guardar en LecturasConMediaMovil (si quieres mantener TopLeft etc se guardan sin modificar)
    rows_mov = []
    for i, t_abs in enumerate(timestamps_uniform):
        # copx_mov[i] puede ser NaN para los primeros MOVING_WINDOW-1 índices
        valx = float(copx_mov[i]) if not np.isnan(copx_mov[i]) else None
        valy = float(copy_mov[i]) if not np.isnan(copy_mov[i]) else None
        rows_mov.append((
            usuarioId,
            numeroPruebas,
            float(interp_data['TopLeft'][i]),
            float(interp_data['TopRight'][i]),
            float(interp_data['BottomLeft'][i]),
            float(interp_data['BottomRight'][i]),
            valx,
            valy,
            float(interp_data['Total'][i]),
            t_abs
        ))
    if rows_mov:
        insert_many(conn, "LecturasConMediaMovil",
                    ["UsuarioId","NumeroPruebas","TopLeft","TopRight","BottomLeft","BottomRight","COP_X","COP_Y","Total","TimeStamp"],
                    rows_mov)
    print(f"  -> Media móvil aplicada e insertadas {len(rows_mov)} filas en LecturasConMediaMovil.")

    # 6) CALCULAR METRICAS para evaluación
    # Debemos excluir 1s inicial (arranque del filtro) y 1s final (media móvil incompleta)
    total_samples = len(t_uniform)
    start_idx = int(math.ceil(FS * 1.0))          # 1 s inicial -> 100 muestras
    end_idx = total_samples - int(math.ceil(FS * 1.0))  # excluir último 1 s

    if end_idx - start_idx <= 0:
        print("  -> Señal demasiado corta tras exclusiones. No se calculan métricas.")
        return

    # tomamos los valores de la señal *con media movil* (porque las métricas se basan en ello)
    x_window = copx_mov[start_idx:end_idx]
    y_window = copy_mov[start_idx:end_idx]

    # limpiar NaNs (puede quedar NAN si por ajuste la ventana no completa al inicio)
    valid_mask = ~np.isnan(x_window) & ~np.isnan(y_window)
    if not np.any(valid_mask):
        print("  -> No hay datos válidos en la ventana final (NaNs). Omitiendo métricas.")
        return

    x_valid = x_window[valid_mask]
    y_valid = y_window[valid_mask]
    # Mean COP
    mean_x = float(np.mean(x_valid))
    mean_y = float(np.mean(y_valid))
    # Longitud de trayectoria (sum distancia euclidiana entre puntos consecutivos)
    dx = np.diff(x_valid)
    dy = np.diff(y_valid)
    path_length = float(np.sum(np.sqrt(dx*dx + dy*dy)))
    # Area del rectángulo
    area_rect = float((np.max(x_valid) - np.min(x_valid)) * (np.max(y_valid) - np.min(y_valid)))
    # RMS (de la magnitud)
    mag = np.sqrt(x_valid**2 + y_valid**2)
    rms = float(np.sqrt(np.mean(mag**2)))

    # insertar en Evaluaciones
    now_ts = timestamps_uniform[start_idx]  # timestamp representativo; o DateTime.now()
    eval_row = [(usuarioId, numeroPruebas, mean_x, mean_y, path_length, area_rect, rms, now_ts)]
    insert_many(conn, "Evaluaciones",
                ["UsuarioId","NumeroPruebas","Mean_COPX","Mean_COPY","LongitudTrayectoria","AreaRectangulo","RMS","TimeStamp"],
                eval_row)
    print(f"  -> Evaluación guardada: MeanX={mean_x:.4f} MeanY={mean_y:.4f} Path={path_length:.4f} RMS={rms:.4f} Area={area_rect:.6f}")

# ========== Main ==========
def main():
    conn = connect()
    try:
        groups = read_groups(conn)
        if not groups:
            print("No hay datos en LecturasBalanceBoard.")
            return
        for (uid, num, df) in groups:
            try:
                process_group(conn, uid, num, df)
            except Exception as e:
                print(f"Error procesando Usuario {uid} Prueba {num}: {e}", file=sys.stderr)
    finally:
        conn.close()
        print("\nPipeline finalizado.")

if __name__ == "__main__":
    main()



Procesando Usuario=6, Prueba=1, muestras originales=2286

Procesando Usuario=6, Prueba=2, muestras originales=1551

Pipeline finalizado.


C:\Users\alexs\AppData\Local\Temp\ipykernel_17840\1986464869.py:54: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
Error procesando Usuario 6 Prueba 1: ('Unknown object type numpy.int64 during describe', 'HY000')
c:\Anaconda\Lib\site-packages\scipy\interpolate\_interpolate.py:712: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
Error procesando Usuario 6 Prueba 2: ('Unknown object type numpy.int64 during describe', 'HY000')
